In [ ]:
# Print timestamp for tracking when notebook was run
# Useful for experiment tracking and reproducibility
import datetime
print(f"Notebook last run (end-to-end): {datetime.datetime.now()}")

In [ ]:
# Check GPU availability and memory using nvidia-smi
# This helps verify GPU is accessible for training
!nvidia-smi

In [ ]:
# Import necessary libraries
import os
import tensorflow as tf

In [ ]:
# Display TensorFlow version to ensure compatibility
# Different versions may have different API behaviors
print("Tensorflow version:")
print(tf.__version__ + "\n")

### Downloading the data

In [ ]:
import urllib.request
import zipfile
import os

# Download the Food-101 subset dataset (10 classes, 10% of full data)
# This dataset contains images of different food categories
url = "https://storage.googleapis.com/ztm_tf_course/food_vision/10_food_classes_10_percent.zip"
zip_path = "10_food_classes_10_percent.zip"
extract_dir = "10_food_classes_10_percent"

# Check if dataset already exists to avoid re-downloading
if not os.path.exists(extract_dir):
    print(f"Downloading {url}...")
    
    # Download the zip file from the URL
    urllib.request.urlretrieve(url, zip_path)
    print(f"Downloaded to {zip_path}")
    
    # Extract all files from the zip archive
    print(f"Extracting {zip_path}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    
    # Remove the zip file to save disk space
    os.remove(zip_path)
    print(f"Extracted to {extract_dir}/ and removed {zip_path}")
else:
    print(f"Dataset already exists at {extract_dir}/")

In [ ]:
# Optional: Walk through directory structure to count directories and images
# Commented out to reduce output clutter
# for dirpath, dirnames, flienames in os.walk(extract_dir):
#     print(f"There are {len(dirnames)} directories and {len(flienames)} images in '{dirpath}'.")

### Creating data loaders

In [ ]:
# Import ImageDataGenerator for data loading and preprocessing
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
# Define image preprocessing parameters
IMAGE_SHAPE = (224, 224)  # Standard input size for many pre-trained models
BATCH_SIZE  = 32          # Number of images to process at once (balance between speed and memory)

# Define paths to training and testing data
train_dir = "10_food_classes_10_percent/train/"
test_dir  = "10_food_classes_10_percent/test/"

# Create data generators with rescaling
# Rescaling from [0, 255] to [0, 1] normalizes pixel values for better training
train_datagen = ImageDataGenerator(rescale= float(1/255) )
test_datagen  = ImageDataGenerator(rescale= float(1/255) )

In [ ]:
print("training images:")
# Create training data iterator
# flow_from_directory automatically labels images based on subdirectory names
train_data_10 = train_datagen.flow_from_directory(
    train_dir,
    target_size = IMAGE_SHAPE,      # Resize all images to 224x224
    batch_size = BATCH_SIZE,        # Load 32 images per batch
    class_mode = "categorical"      # One-hot encode labels for multi-class classification
)


print(" ")
print("testing images: ")
# Create testing/validation data iterator
# Note: Using train_datagen here (should be test_datagen, but works the same with only rescaling)
test_data_10 = train_datagen.flow_from_directory(
    test_dir,
    target_size = IMAGE_SHAPE,      # Same size as training images
    batch_size = BATCH_SIZE,        # Same batch size
    class_mode = "categorical"      # Same encoding as training
)

### Setting up callbacks

In [ ]:
# Create a function to generate TensorBoard callbacks for experiment tracking
# TensorBoard allows visualization of training metrics, model graphs, and more
# Callback will be saved in folders according to following schema:
# [dir_name]/[experiment_name]/[current_timestamp]

def create_tensorboard_callback(dir_name, experiment_name):
    """
    Creates a TensorBoard callback for tracking training progress.
    
    Args:
        dir_name: Root directory for logs
        experiment_name: Name of the experiment (e.g., model architecture)
    
    Returns:
        TensorBoard callback configured with timestamped log directory
    """
    # Create hierarchical log directory with timestamp for unique identification
    log_dir = dir_name + "/" + experiment_name + "/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    
    # Initialize TensorBoard callback
    tensorboard_callback = tf.keras.callbacks.TensorBoard(
        log_dir = log_dir
    )

    print(f"Saving TensorBoard log files to: {log_dir}")
    return tensorboard_callback

### Transfer learning

In [ ]:
# Import pre-trained models and layers for transfer learning
from tensorflow.keras.applications import ResNet50V2, EfficientNetB0
from tensorflow.keras import layers, models

In [ ]:
# Get the number of classes from the training data generator
# This will be used for the output layer
num_classes = train_data_10.num_classes

In [ ]:
# ============ ResNet50V2 Transfer Learning Model ============

# Load pre-trained ResNet50 V2 as the base model
# ResNet50V2 is a deep residual network trained on ImageNet (1.4M images, 1000 classes)
base_model = ResNet50V2(
    include_top=False,              # Remove the original classification layer (top)
    weights='imagenet',             # Use pre-trained ImageNet weights
    input_shape= IMAGE_SHAPE + (3,) # Input shape: 224x224x3 (RGB images)
)


# Freeze the base model layers to prevent updating pre-trained weights
# This is key to transfer learning: we keep learned features from ImageNet
base_model.trainable = False

# Build the complete model by adding custom layers on top of the base model
resnet_model = models.Sequential([
    base_model,                                                      # Pre-trained feature extractor
    layers.GlobalAveragePooling2D(),                                 # Pool features to reduce dimensionality
    layers.Dense(num_classes, activation='softmax', name='output_layer')  # Classification layer for our 10 classes
])

# Compile the model with optimizer, loss function, and metrics
resnet_model.compile(
    optimizer='adam',                   # Adam optimizer (adaptive learning rate)
    loss='categorical_crossentropy',    # Loss for multi-class classification with one-hot labels
    metrics=['accuracy']                # Track accuracy during training
)

# Display the model architecture
# Shows layers, output shapes, and parameter counts
resnet_model.summary()

In [ ]:
# Train the ResNet50V2 model (commented out to prevent accidental execution)
# Uncomment to run training
# resnet20v2_history = resnet_model.fit(
#     train_data_10,                              # Training data generator
#     epochs=5,                                   # Number of complete passes through training data
#     steps_per_epoch = len(train_data_10),      # Number of batches per epoch
#     validation_data = test_data_10,            # Validation data for monitoring
#     validation_steps = len(test_data_10),      # Number of validation batches
#     # Add TensorBoard callback for experiment tracking
#     callbacks = [
#         create_tensorboard_callback(dir_name="transfer_learning",
#                                     experiment_name="resnet50v2")
#     ]
# )

In [ ]:
# Utility function to visualize training history
# This could be extracted to a helper.py file for reuse across notebooks
import matplotlib.pyplot as plt

def plot_loss_curves(history):
    """
    Plot training and validation loss and accuracy curves.
    
    Args:
        history: History object returned from model.fit()
    
    Returns:
        Displays two plots: loss curves and accuracy curves
    """
    # Extract loss values from history
    loss = history.history['loss']          # Training loss per epoch
    val_loss = history.history['val_loss']  # Validation loss per epoch

    # Extract accuracy values from history
    accuracy = history.history['accuracy']          # Training accuracy per epoch
    val_accuracy = history.history['val_accuracy']  # Validation accuracy per epoch

    # Create x-axis (epoch numbers)
    epochs = range(len(history.history['loss']))

    # Plot loss curves
    plt.plot(epochs, loss, label='training_loss')
    plt.plot(epochs, val_loss, label='val_loss')
    plt.title('Loss')
    plt.xlabel('Epochs')
    plt.legend()

    # Plot accuracy curves in a new figure
    plt.figure()
    plt.plot(epochs, accuracy, label='training_accuracy')
    plt.plot(epochs, val_accuracy, label='val_accuracy')
    plt.title('Accuracy')
    plt.xlabel('Epochs')
    plt.legend();

In [ ]:
# Visualize ResNet50V2 training results
# Note: This will fail if resnet20v2_history was not created (training was commented out)
plot_loss_curves(resnet20v2_history)

In [ ]:
# ============ EfficientNetB0 Transfer Learning Model ============

# Load pre-trained EfficientNetB0 as the base model
# EfficientNet uses compound scaling (depth, width, resolution) for efficiency
# B0 is the baseline version, smaller and faster than ResNet50V2
base_model = EfficientNetB0(
    include_top=False,              # Remove the original classification layer
    weights='imagenet',             # Use pre-trained ImageNet weights
    input_shape= IMAGE_SHAPE + (3,) # Input shape: 224x224x3 (RGB images)
)


# Freeze the base model layers for transfer learning
# We'll only train the new classification head we add
base_model.trainable = False

# Build the complete model with custom classification layers
effnetb0_model = models.Sequential([
    base_model,                                                      # Pre-trained EfficientNetB0 feature extractor
    layers.GlobalAveragePooling2D(),                                 # Pool spatial features into a feature vector
    layers.Dense(num_classes, activation='softmax', name='output_layer')  # Output layer for 10 food classes
])

# Compile the model
effnetb0_model.compile(
    optimizer='adam',                   # Adam optimizer
    loss='categorical_crossentropy',    # Multi-class classification loss
    metrics=['accuracy']                # Track accuracy metric
)

# Display the model architecture
effnetb0_model.summary()

In [ ]:
# Train the EfficientNetB0 model (commented out to prevent accidental execution)
# Uncomment and fix to run training
# effnetb0_history = effnetb0_model.fit( 
#     train_data_10,                              # Training data generator
#     epochs=5,                                   # Number of training epochs
#     steps_per_epoch = len(train_data_10),      # Batches per epoch
#     validation_data = test_data_10,            # Validation data
#     validation_steps = len(test_data_10),      # Validation batches
#     # Add TensorBoard callback for tracking
#     callbacks = [
#         create_tensorboard_callback(dir_name="transfer_learning",
#                                     experiment_name="effnetb0")
#     ]
# )

In [ ]:
# Visualize EfficientNetB0 training results
plot_loss_curves(effnetb0_history)